*0.1 Python for GenAI*

# structlog

**The situation.** A million log lines a day. Someone asks: "which users had requests slower than two seconds this morning?" With plain sentences the answer is a regular expression, a spreadsheet and an afternoon — and half the lines are missing the request id that would tie them together.

**The fix: log fields, not sentences.** `structlog` writes each event as a small JSON record with named fields. A log system (Datadog, Loki, CloudWatch) indexes the fields, so that question becomes one query: `seconds > 2 and user_id = "u-42"`. And a request's context — its id, its user — is bound once and appears on every line automatically.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Configure once; bind per request; log events.** `bind_contextvars` attaches the request's identity to everything logged afterwards.

In [2]:
import io
import json
import time
import uuid

import structlog
from openai import OpenAI

buffer = io.StringIO()
structlog.configure(
    processors=[
        structlog.contextvars.merge_contextvars,  # attach the bound request context
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer(),
    ],
    logger_factory=structlog.PrintLoggerFactory(file=buffer),
)
log = structlog.get_logger("support.chat")

# One request arrives: bind its identity once.
structlog.contextvars.clear_contextvars()
structlog.contextvars.bind_contextvars(request_id=uuid.uuid4().hex[:8], user_id="u-42", model=MODEL)

client = OpenAI(timeout=30)
started = time.perf_counter()
log.info("completion.start", question_chars=28)
reply = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "Say bye in one word."}], temperature=0
)
log.info(
    "completion.end",
    seconds=round(time.perf_counter() - started, 3),
    prompt_tokens=reply.usage.prompt_tokens,
    completion_tokens=reply.usage.completion_tokens,
)

events = []
for line in buffer.getvalue().splitlines():
    events.append(json.loads(line))
print(json.dumps(events, indent=2))
assert events[0]["request_id"] == events[1]["request_id"]

[
  {
    "question_chars": 28,
    "event": "completion.start",
    "model": "gpt-4o-mini",
    "request_id": "946a09da",
    "user_id": "u-42",
    "level": "info",
    "timestamp": "2026-09-21T15:50:53.875356Z"
  },
  {
    "seconds": 0.847,
    "prompt_tokens": 13,
    "completion_tokens": 3,
    "event": "completion.end",
    "model": "gpt-4o-mini",
    "request_id": "946a09da",
    "user_id": "u-42",
    "level": "info",
    "timestamp": "2026-09-21T15:50:54.722866Z"
  }
]


**Reading the output.** Two JSON records. Both carry `request_id`, `user_id` and `model` — bound once, never repeated in the calls. The second has `seconds` and token counts as numbers, so a log system can compare and graph them.

```
plain       done in 0.42s tokens=15                                       → grep and guess
structured  {"event": "completion.end", "seconds": 0.42, "tokens": 15,
             "request_id": "a1b2", "user_id": "u-42"}                       → filter · group · alert
```

**The rule to remember.** Same event names everywhere (`completion.start`, `completion.end`, `tool.call`); numbers as numbers; ids bound once per request.

| Use it when | Don't when | Instead use |
|---|---|---|
| logs go to a search system | a single developer reading a terminal | plain logging |

**Watch out**
- Clear the bound context at the start of each request. Long-running workers otherwise log the previous request's user id.
- Do not log full prompts and answers: cost, privacy, and index size.
- A new *value* per request (`request_id`) is fine; a new *field name* per request breaks indexing.